# RetinaVessel AI — controlled training v2

This notebook repeats all three seed-42 development experiments under one declared policy: a maximum of 30 epochs with patience-5 early stopping. It writes exact-resume checkpoints after every epoch and packages the complete development artifacts.

**Protocol boundary:** this notebook uses only the frozen FIVES training/validation manifest. It does not evaluate, inspect, or tune against the official FIVES test set.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/simlexx-k/retinavessel-ai.git"
EXPECTED_COMMIT = "0c40aa665be3a247372062eaa46a33f578cf7b0f"
REPO = Path("/kaggle/working/retinavessel-ai-v2-code")

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)

subprocess.run(
    ["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_COMMIT],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")],
    check=True,
)
os.chdir(REPO)
actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
).strip()
assert actual_commit == EXPECTED_COMMIT
print("Pinned training code:", actual_commit)
print("Working directory:", Path.cwd())

Cloning into '/kaggle/working/retinavessel-ai-v2-code'...
HEAD is now at 0c40aa6 Add resumable training and early stopping


Pinned training code: 0c40aa665be3a247372062eaa46a33f578cf7b0f
Working directory: /kaggle/working/retinavessel-ai-v2-code


In [2]:
import json
import torch

sys.path.insert(0, str(REPO / "src"))
from retinavessel.data import find_fives_root, load_manifest

DATA_MOUNT = Path("/kaggle/input/datasets/simonkosgei/fives-retinal-dataset")
SPLIT_MANIFEST = REPO / "splits/fives_seed42.json"
assert DATA_MOUNT.exists(), f"Missing Kaggle dataset: {DATA_MOUNT}"
assert torch.cuda.is_available(), "Enable a GPU accelerator before running v2"

FIVES_ROOT = find_fives_root(DATA_MOUNT)
official_train = load_manifest(FIVES_ROOT, "train")
official_test = load_manifest(FIVES_ROOT, "test")
split = json.loads(SPLIT_MANIFEST.read_text())
assert len(official_train) == 600
assert len(official_test) == 200
assert len(split["train"]) == 504
assert len(split["validation"]) == 89
assert len(split["excluded"]) == 7

print("PyTorch:", torch.__version__)
print("CUDA device:", torch.cuda.get_device_name(0))
print("FIVES root:", FIVES_ROOT)
print("Frozen development split: 504 train / 89 validation / 7 excluded")
print("Official test count verified but not evaluated: 200")
subprocess.run([
    "nvidia-smi",
    "--query-gpu=name,memory.total,driver_version",
    "--format=csv",
], check=True)

PyTorch: 2.11.0+cu130
CUDA device: Tesla T4
FIVES root: /kaggle/input/datasets/simonkosgei/fives-retinal-dataset/FIVES A Fundus Image Dataset for AI-based Vessel Segmentation
Frozen development split: 504 train / 89 validation / 7 excluded
Official test count verified but not evaluated: 200
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], returncode=0)

In [3]:
test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = str(REPO / "src")
subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=REPO,
    env=test_environment,
    check=True,
)

.....                                                                    [100%]
5 passed in 7.79s


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)

## Controlled experiment configuration

All models use the same frozen split, seed, patch sampling budget, batch size, optimizer defaults, maximum epoch count, and early-stopping patience. Existing v1 checkpoints are intentionally not reused because they lack optimizer and random-number-generator state.

In [4]:
import shlex
import time

MAX_EPOCHS = 30
PATIENCE = 5
SEED = 42
RUN_TIMES = {}

EXPERIMENTS = [
    {"name": "unet_seed42_es", "model": "unet", "degradation": False},
    {"name": "lite_unet_seed42_es", "model": "lite_unet", "degradation": False},
    {"name": "lite_robust_seed42_es", "model": "lite_unet", "degradation": True},
]

def run_experiment(spec):
    output = REPO / "outputs" / spec["name"]
    last_checkpoint = output / "last.pt"
    command = [
        sys.executable, "-m", "retinavessel.train",
        "--data", str(DATA_MOUNT),
        "--model", spec["model"],
        "--epochs", str(MAX_EPOCHS),
        "--patience", str(PATIENCE),
        "--batch-size", "2",
        "--patch-size", "384",
        "--patches-per-image", "2",
        "--split-manifest", str(SPLIT_MANIFEST),
        "--seed", str(SEED),
        "--device", "cuda",
        "--out", str(output),
    ]
    if spec["degradation"]:
        command.append("--degradation")

    if last_checkpoint.exists():
        state = torch.load(last_checkpoint, map_location="cpu", weights_only=False)
        complete = (
            int(state["epoch"]) >= MAX_EPOCHS
            or int(state["epochs_without_improvement"]) >= PATIENCE
        )
        if complete:
            print(f"{spec['name']} already complete at epoch {state['epoch']}; skipping")
            return
        command.extend(["--resume", str(last_checkpoint)])

    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(REPO / "src")
    print("Running:", shlex.join(command))
    started = time.perf_counter()
    subprocess.run(command, cwd=REPO, env=environment, check=True)
    RUN_TIMES[spec["name"]] = time.perf_counter() - started
    for required in ("best.pt", "last.pt", "history.json", "split_manifest.json"):
        assert (output / required).is_file(), f"Missing {output / required}"

## E1 — standard U-Net

In [5]:
run_experiment(EXPERIMENTS[0])

Running: /usr/bin/python3 -m retinavessel.train --data /kaggle/input/datasets/simonkosgei/fives-retinal-dataset --model unet --epochs 30 --patience 5 --batch-size 2 --patch-size 384 --patches-per-image 2 --split-manifest /kaggle/working/retinavessel-ai-v2-code/splits/fives_seed42.json --seed 42 --device cuda --out /kaggle/working/retinavessel-ai-v2-code/outputs/unet_seed42_es
device=cuda, model=unet, params=1,964,097


{'epoch': 1, 'training_loss': 0.9521474115077465, 'validation_dice': 0.41079425152779653}


{'epoch': 2, 'training_loss': 0.670160038840203, 'validation_dice': 0.583151430299942}


{'epoch': 3, 'training_loss': 0.5564039270023978, 'validation_dice': 0.6397625791757829}


{'epoch': 4, 'training_loss': 0.5261581069124597, 'validation_dice': 0.7198023620648805}


{'epoch': 5, 'training_loss': 0.47896246881120735, 'validation_dice': 0.7103275788371122}


... (11 outputs left out)

{'epoch': 17, 'training_loss': 0.33670062274627743, 'validation_dice': 0.794882306135118}


{'epoch': 18, 'training_loss': 0.3278249579496563, 'validation_dice': 0.7965002482728495}


{'epoch': 19, 'training_loss': 0.31876502625112024, 'validation_dice': 0.8016240490482583}


{'epoch': 20, 'training_loss': 0.3188192577855218, 'validation_dice': 0.7880349283055954}
early stopping at epoch 20; best validation Dice=0.823469


## E2 — lightweight U-Net

In [6]:
run_experiment(EXPERIMENTS[1])

Running: /usr/bin/python3 -m retinavessel.train --data /kaggle/input/datasets/simonkosgei/fives-retinal-dataset --model lite_unet --epochs 30 --patience 5 --batch-size 2 --patch-size 384 --patches-per-image 2 --split-manifest /kaggle/working/retinavessel-ai-v2-code/splits/fives_seed42.json --seed 42 --device cuda --out /kaggle/working/retinavessel-ai-v2-code/outputs/lite_unet_seed42_es
device=cuda, model=lite_unet, params=236,156


{'epoch': 1, 'training_loss': 0.9200747334767901, 'validation_dice': 0.6137227613661651}


{'epoch': 2, 'training_loss': 0.5372937815116038, 'validation_dice': 0.7048105810201285}


{'epoch': 3, 'training_loss': 0.43670383834886173, 'validation_dice': 0.7461745239534697}


{'epoch': 4, 'training_loss': 0.40713570615838446, 'validation_dice': 0.7985430971262069}


{'epoch': 5, 'training_loss': 0.3779097188352829, 'validation_dice': 0.789015250372995}


... (4 outputs left out)

{'epoch': 10, 'training_loss': 0.3404458989019668, 'validation_dice': 0.8129521343264152}


{'epoch': 11, 'training_loss': 0.34097486460906645, 'validation_dice': 0.8073619048267243}


{'epoch': 12, 'training_loss': 0.3289058390886538, 'validation_dice': 0.7899602522962182}


{'epoch': 13, 'training_loss': 0.32035245023490416, 'validation_dice': 0.8254912357978428}
early stopping at epoch 13; best validation Dice=0.826155


## E3 — lightweight U-Net with training-only degradation

In [7]:
run_experiment(EXPERIMENTS[2])

Running: /usr/bin/python3 -m retinavessel.train --data /kaggle/input/datasets/simonkosgei/fives-retinal-dataset --model lite_unet --epochs 30 --patience 5 --batch-size 2 --patch-size 384 --patches-per-image 2 --split-manifest /kaggle/working/retinavessel-ai-v2-code/splits/fives_seed42.json --seed 42 --device cuda --out /kaggle/working/retinavessel-ai-v2-code/outputs/lite_robust_seed42_es --degradation
device=cuda, model=lite_unet, params=236,156


{'epoch': 1, 'training_loss': 0.9676315575719826, 'validation_dice': 0.5959213041775119}


{'epoch': 2, 'training_loss': 0.5733898180049091, 'validation_dice': 0.7161696878062698}


{'epoch': 3, 'training_loss': 0.46191536729770993, 'validation_dice': 0.7274153525171654}


{'epoch': 4, 'training_loss': 0.43527949421060463, 'validation_dice': 0.7857531028910609}


{'epoch': 5, 'training_loss': 0.39846179965469575, 'validation_dice': 0.7657596815710693}


... (6 outputs left out)

{'epoch': 12, 'training_loss': 0.34770861065517816, 'validation_dice': 0.8285765089596839}


{'epoch': 13, 'training_loss': 0.3364363422558185, 'validation_dice': 0.8181595642667029}


{'epoch': 14, 'training_loss': 0.32752897937796893, 'validation_dice': 0.8374164789710774}


{'epoch': 15, 'training_loss': 0.323769402808495, 'validation_dice': 0.7780835027456332}
early stopping at epoch 15; best validation Dice=0.845217


## Development-set summary and provenance

In [8]:
import hashlib
import pandas as pd

from retinavessel.model import UNet, model_params

rows = []
for spec in EXPERIMENTS:
    output = REPO / "outputs" / spec["name"]
    history = json.loads((output / "history.json").read_text())
    checkpoint = torch.load(output / "best.pt", map_location="cpu", weights_only=True)
    network = UNet(
        base=int(checkpoint["base"]),
        lite=checkpoint["architecture"] == "lite_unet",
    )
    rows.append({
        "experiment": spec["name"],
        "model": checkpoint["architecture"],
        "degradation_training": spec["degradation"],
        "parameters": model_params(network),
        "completed_epochs": history[-1]["epoch"],
        "best_epoch": checkpoint["epoch"],
        "best_validation_dice": checkpoint["validation_dice"],
        "wall_minutes_this_notebook": round(RUN_TIMES.get(spec["name"], float("nan")) / 60, 2),
    })

summary = pd.DataFrame(rows).sort_values("best_validation_dice", ascending=False)
display(summary)

SUMMARY_DIR = REPO / "outputs" / "v2_summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(SUMMARY_DIR / "validation_summary.csv", index=False)
run_manifest = {
    "training_code_commit": EXPECTED_COMMIT,
    "dataset_mount": str(DATA_MOUNT),
    "resolved_fives_root": str(FIVES_ROOT),
    "split_manifest_sha256": hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest(),
    "pytorch_version": torch.__version__,
    "cuda_device": torch.cuda.get_device_name(0),
    "maximum_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "seed": SEED,
    "official_test_evaluated": False,
    "results": json.loads(summary.to_json(orient="records")),
}
(SUMMARY_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
print("Summary written to", SUMMARY_DIR)

,experiment,model,degradation_training,parameters,completed_epochs,best_epoch,best_validation_dice,wall_minutes_this_notebook
2,lite_robust_seed42_es,lite_unet,True,236156,15,10,0.845217,79.90
1,lite_unet_seed42_es,lite_unet,False,236156,13,8,0.826155,68.18
0,unet_seed42_es,unet,False,1964097,20,15,0.823469,127.41


Summary written to /kaggle/working/retinavessel-ai-v2-code/outputs/v2_summary


In [9]:
ARCHIVE = Path("/kaggle/working/retinavessel_v2_seed42.zip")
if ARCHIVE.exists():
    ARCHIVE.unlink()
subprocess.run(
    [
        "zip", "-r", str(ARCHIVE),
        "outputs/unet_seed42_es",
        "outputs/lite_unet_seed42_es",
        "outputs/lite_robust_seed42_es",
        "outputs/v2_summary",
    ],
    cwd=REPO,
    check=True,
)
print(f"Artifact archive: {ARCHIVE} ({ARCHIVE.stat().st_size / 1024**2:.1f} MiB)")
print("Use Kaggle Save Version / Save & Run All, then download this archive.")

  adding: outputs/unet_seed42_es/ (stored 0%)
  adding: outputs/unet_seed42_es/split_manifest.json (deflated 76%)
  adding: outputs/unet_seed42_es/history.json (deflated 75%)
  adding: outputs/unet_seed42_es/best.pt (deflated 8%)
  adding: outputs/unet_seed42_es/last.pt (deflated 8%)
  adding: outputs/lite_unet_seed42_es/ (stored 0%)
  adding: outputs/lite_unet_seed42_es/split_manifest.json (deflated 76%)
  adding: outputs/lite_unet_seed42_es/history.json (deflated 73%)
  adding: outputs/lite_unet_seed42_es/best.pt (deflated 10%)
  adding: outputs/lite_unet_seed42_es/last.pt (deflated 10%)
  adding: outputs/lite_robust_seed42_es/ (stored 0%)
  adding: outputs/lite_robust_seed42_es/split_manifest.json (deflated 76%)
  adding: outputs/lite_robust_seed42_es/history.json (deflated 74%)
  adding: outputs/lite_robust_seed42_es/best.pt (deflated 10%)
  adding: outputs/lite_robust_seed42_es/last.pt (deflated 10%)
  adding: outputs/v2_summary/ (stored 0%)
  adding: outputs/v2_summary/run_manife

## Stop here

Do not run the official FIVES test evaluation from this notebook. The next stage is validation-only robustness, topology, latency, and peak-memory analysis, followed by multi-seed replication. The official test remains sealed until those decisions are frozen.